# Deep tier on CSI-HAR: do the two walls hold on the second dataset?

Runs the five standard deep architectures (CNN, BiGRU, Transformer, Chebyshev-KAN,
lightweight SSM) on **CSI-HAR** ($T=64$, $F=52$), reporting subject-independent
(leave-one-user-out) accuracy, parameter count, int8 convertibility, and int8 size.
Convertible models are saved as int8 TFLite so they can be flashed to the classic ESP32
to test the memory wall on this dataset (CSI-HAR has a smaller feature dimension than
UT-HAR, 52 vs 90, so its arenas are smaller).

Attach **sayakghorai34/csi-har-dataset**. Settings: GPU T4, Internet On.


In [1]:
import os, json, warnings
from pathlib import Path
import numpy as np, tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings("ignore")
print("TensorFlow", tf.__version__)
OUT=Path("/kaggle/working"); (OUT/"tflite").mkdir(parents=True,exist_ok=True)
TARGET_T=64; EPOCHS=40


2026-06-07 19:11:51.597257: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780859511.813043      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780859511.876495      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780859512.368178      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780859512.368221      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780859512.368224      22 computation_placer.cc:177] computation placer alr

TensorFlow 2.19.0


In [2]:
# ----- CSI-HAR loader (subject-independent leave-one-user-out) -----
import re
def find_csi_har_root():
    for c in Path("/kaggle/input").rglob("CSI-HAR-Dataset"):
        if c.is_dir(): return c
    for c in Path("/kaggle/input").rglob("*"):
        if c.is_dir() and (c/"walk").is_dir() and (c/"run").is_dir(): return c
    return None
def load_csi_har_raw(T=TARGET_T):
    root=find_csi_har_root()
    if root is None: raise FileNotFoundError("CSI-HAR-Dataset not attached")
    files=[p for p in root.rglob("*_A.csv") if not p.name.startswith("Annotation")]
    acts=sorted({p.parent.name for p in files}); lmap={a:i for i,a in enumerate(acts)}
    X=[];y=[];users=[]
    for p in files:
        try: a=np.genfromtxt(str(p),delimiter=",")
        except Exception: continue
        if a.ndim==1: a=a.reshape(-1,1)
        if a.shape[0]<2 or a.shape[1]<2: continue
        idx=np.linspace(0,a.shape[0]-1,T).astype(int)
        X.append(a[idx,:].astype(np.float32)); y.append(lmap[p.parent.name])
        m=re.search(r"user_(\d+)_",p.name); users.append(int(m.group(1)) if m else 0)
    X=np.asarray(X,np.float32); y=np.asarray(y,np.int64); users=np.asarray(users)
    m=X.mean(axis=(1,2),keepdims=True); s=X.std(axis=(1,2),keepdims=True)+1e-8; X=((X-m)/s).astype(np.float32)
    print(f"CSI-HAR {X.shape} F={X.shape[2]} classes={acts} users={sorted(set(users.tolist()))}")
    return X,y,users
def louo_folds():
    X,y,u=load_csi_har_raw(); n=int(y.max())+1; folds=[]
    for uu in sorted(set(u.tolist())):
        te=u==uu; folds.append((X[~te],y[~te],X[te],y[te]))
    return folds,n


In [3]:
# ----- Chebyshev-KAN + diagonal SSM (same as the 5-arch benchmark) -----
class ChebyKAN(layers.Layer):
    def __init__(self,out_dim,degree=3,**kw): super().__init__(**kw); self.out_dim=out_dim; self.degree=degree
    def build(self,shape):
        self.in_dim=int(shape[-1])
        self.coeff=self.add_weight(name="coeff",shape=(self.in_dim*(self.degree+1),self.out_dim),
            initializer=tf.keras.initializers.GlorotUniform())
    def call(self,x):
        x=tf.tanh(x); Ts=[tf.ones_like(x),x]
        for k in range(2,self.degree+1): Ts.append(2.0*x*Ts[-1]-Ts[-2])
        return tf.matmul(tf.concat(Ts,axis=-1),self.coeff)
class DiagSSMCell(layers.Layer):
    def __init__(self,units,**kw): super().__init__(**kw); self.units=units; self.state_size=units
    def build(self,shape):
        d=int(shape[-1])
        self.A=self.add_weight(name="A",shape=(self.units,),initializer=tf.keras.initializers.RandomUniform(-0.99,0.99))
        self.B=self.add_weight(name="B",shape=(d,self.units),initializer="glorot_uniform")
        self.C=self.add_weight(name="C",shape=(self.units,d),initializer="glorot_uniform")
    def call(self,inputs,states):
        h=states[0]*tf.tanh(self.A)+tf.matmul(inputs,self.B); return tf.matmul(h,self.C),[h]

def m_cnn(T,F,n):
    i=layers.Input((T,F)); x=layers.Conv1D(64,7,padding="same",activation="relu")(i)
    x=layers.BatchNormalization()(x); x=layers.MaxPool1D(2)(x)
    x=layers.Conv1D(128,5,padding="same",activation="relu")(x)
    x=layers.BatchNormalization()(x); x=layers.GlobalAveragePooling1D()(x)
    x=layers.Dense(64,activation="relu")(x); return Model(i,layers.Dense(n)(x),name="CNN")
def m_gru(T,F,n):
    i=layers.Input((T,F)); x=layers.Bidirectional(layers.GRU(64))(i)
    x=layers.Dense(64,activation="relu")(x); return Model(i,layers.Dense(n)(x),name="BiGRU")
def m_transformer(T,F,n):
    i=layers.Input((T,F)); x=layers.Conv1D(64,5,padding="same")(i)
    a=layers.MultiHeadAttention(num_heads=4,key_dim=16)(x,x); x=layers.LayerNormalization()(x+a)
    f=layers.Dense(128,activation="relu")(x); f=layers.Dense(64)(f); x=layers.LayerNormalization()(f)
    x=layers.GlobalAveragePooling1D()(x); return Model(i,layers.Dense(n)(x),name="Transformer")
def m_kan(T,F,n):
    i=layers.Input((T,F)); x=layers.Conv1D(64,7,padding="same",activation="relu")(i)
    x=layers.MaxPool1D(2)(x); x=layers.Conv1D(64,5,padding="same",activation="relu")(x)
    x=layers.GlobalAveragePooling1D()(x); x=ChebyKAN(64,degree=3)(x); return Model(i,ChebyKAN(n,degree=3)(x),name="ChebyKAN")
def m_ssm(T,F,n):
    i=layers.Input((T,F)); x=layers.Conv1D(64,5,padding="same",activation="relu")(i)
    x=layers.RNN(DiagSSMCell(64))(x); x=layers.Dense(64,activation="relu")(x); return Model(i,layers.Dense(n)(x),name="SSM")
BUILDERS={"CNN":m_cnn,"BiGRU":m_gru,"Transformer":m_transformer,"ChebyKAN":m_kan,"SSM":m_ssm}


In [4]:
# ----- train (no string metric: TF-2.19 Keras-3 raises dtype='string' otherwise) -----
def train_once(builder,Xtr,ytr,Xte,yte,n,seed=0):
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    m=builder(int(Xtr.shape[1]),int(Xtr.shape[2]),int(n))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    m.fit(np.asarray(Xtr,np.float32),np.asarray(ytr,np.int64),epochs=EPOCHS,batch_size=64,verbose=0)
    acc=float(accuracy_score(yte,m.predict(Xte,verbose=0).argmax(-1)))
    return m,acc,int(m.count_params())

def to_int8(model,Xtr,name):
    def rep():
        for i in range(min(300,len(Xtr))): yield [Xtr[i:i+1].astype(np.float32)]
    last=None
    for src in ["keras","savedmodel"]:
        for io in ["int8","float"]:
            try:
                if src=="keras": c=tf.lite.TFLiteConverter.from_keras_model(model)
                else:
                    d=str(OUT/"sm"/name)
                    try: model.export(d)
                    except Exception: tf.saved_model.save(model,d)
                    c=tf.lite.TFLiteConverter.from_saved_model(d)
                c.optimizations=[tf.lite.Optimize.DEFAULT]; c.representative_dataset=rep
                c.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8,tf.lite.OpsSet.TFLITE_BUILTINS]
                if io=="int8": c.inference_input_type=tf.int8; c.inference_output_type=tf.int8
                blob=c.convert(); (OUT/"tflite"/f"{name}_csihar_int8.tflite").write_bytes(blob)
                return len(blob)
            except Exception as e: last=e
    raise last


In [5]:
import pandas as pd
folds,N=louo_folds()
rows=[]; results={}
for name,b in BUILDERS.items():
    accs=[]; last=None; lastX=None; params=0
    for (Xtr,ytr,Xte,yte) in folds:
        m,acc,params=train_once(b,Xtr,ytr,Xte,yte,N); accs.append(acc); last=m; lastX=Xtr
    try: sz=to_int8(last,lastX,name); conv=True; kb=round(sz/1024,1)
    except Exception as e: conv=False; kb=-1
    results[name]=dict(acc_mean=float(np.mean(accs)),acc_std=float(np.std(accs)),params=int(params),converts=conv,int8_kb=kb)
    rows.append({"Model":name,"Acc(%)":f"{np.mean(accs)*100:.2f} +/- {np.std(accs)*100:.2f}","Params(K)":round(params/1e3,1),
                 "Converts":"yes" if conv else "no","int8(kB)":kb if kb>0 else "n/a"})
    print(f"  {name:12s} acc {np.mean(accs)*100:.2f}  params {params/1e3:.1f}K  converts {conv}  int8 {kb}kB")
tbl=pd.DataFrame(rows); print("\nDeep tier on CSI-HAR (subject-independent)\n"+"-"*60); print(tbl.to_string(index=False))
tbl.to_csv(OUT/"deeptier_csihar_table.csv",index=False)
json.dump(results,open(OUT/"deeptier_csihar_results.json","w"),indent=2)
print("\nsaved deeptier_csihar_table.csv, deeptier_csihar_results.json, tflite/*_csihar_int8.tflite")
print("Convertible models -> flash to the classic ESP32 to test the memory wall on CSI-HAR.")
tbl


CSI-HAR (420, 64, 52) F=52 classes=['bend', 'fall', 'lie down', 'run', 'sitdown', 'standup', 'walk'] users=[1, 2, 3]


I0000 00:00:1780859539.428517      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1780859542.592681      64 service.cc:152] XLA service 0x7bb1e000ca00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780859542.592735      64 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1780859542.996981      64 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780859545.295057      64 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


INFO:tensorflow:Assets written to: /tmp/tmpcq5au25m/assets


INFO:tensorflow:Assets written to: /tmp/tmpcq5au25m/assets


Saved artifact at '/tmp/tmpcq5au25m'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 52), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  136007193292880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136007193289808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136007193294224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136007193293456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136007193293648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136007193291152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136007187732368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136007187736400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136007187739088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136007187730640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136007187739664: Te

...

INFO:tensorflow:Assets written to: /kaggle/working/sm/SSM/assets


INFO:tensorflow:Assets written to: /kaggle/working/sm/SSM/assets


Saved artifact at '/kaggle/working/sm/SSM'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 52), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  136003154474960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136003154465360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136003154464976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136003154465936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136003154467280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136003154466128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136003157829456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136003157833104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136003154465552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  SSM          acc 47.38  params 29.6K  converts False  int8 -1kB

Deep tier on CSI-H

W0000 00:00:1780859670.169705      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1780859670.169738      22 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
loc(callsite(callsite(fused["TensorListReserve:", "SSM_1/rnn_1/TensorArrayV2_1@__inference___call___73902"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper___call___73949"]) at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): error: 'tf.TensorListReserve' op requires element_shape to be static during TF Lite transformation pass
loc(callsite(callsite(fused["TensorListReserve:", "SSM_1/rnn_1/TensorArrayV2_1@__inference___call___73902"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper___call___73949"]) at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): error: failed to legalize operation 'tf.TensorListReserve' that was explicitly marked illegal
error: Lowering tensor list 

,Model,Acc(%),Params(K),Converts,int8(kB)
0,CNN,64.05 +/- 4.71,73.9,yes,86.3
1,BiGRU,64.29 +/- 0.58,54.0,no,n/a
2,Transformer,58.10 +/- 3.42,50.6,yes,76.3
3,ChebyKAN,65.71 +/- 3.09,62.1,yes,73.7
4,SSM,47.38 +/- 10.79,29.6,no,n/a


## Next
Download `tflite/*_csihar_int8.tflite` for the models that converted (expected: CNN,
Transformer, Chebyshev-KAN; not BiGRU/SSM, which fail the convertibility wall on any
dataset). Flash each to the classic ESP32 and check whether `AllocateTensors` succeeds
within the ~108 kB budget; this settles the memory wall for CSI-HAR.
